In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [6]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded successfully")
from datasets import load_dataset

dataset = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en"
)

print(dataset)

Libraries loaded successfully
DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
})


In [8]:
def create_sentiment_label(stars):
    if stars <= 2:
        return "negative"
    elif stars == 3:
        return "neutral"
    else:
        return "positive"


def prepare_split(split):
    df = split.to_pandas()
    df["sentiment"] = df["stars"].apply(create_sentiment_label)
    return df


train_df = prepare_split(dataset["train"])
val_df = prepare_split(dataset["validation"])
test_df = prepare_split(dataset["test"])


import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


train_df_clean = train_df.drop_duplicates(
    subset=["review_body", "sentiment"]
).copy()

val_df_clean = val_df.drop_duplicates(
    subset=["review_body", "sentiment"]
).copy()

test_df_clean = test_df.drop_duplicates(
    subset=["review_body", "sentiment"]
).copy()

train_df_clean["text"] = train_df_clean["review_body"].apply(clean_text)
val_df_clean["text"] = val_df_clean["review_body"].apply(clean_text)
test_df_clean["text"] = test_df_clean["review_body"].apply(clean_text)

print("Train:", train_df_clean.shape)
print("Validation:", val_df_clean.shape)
print("Test:", test_df_clean.shape)




Train: (199499, 10)
Validation: (4999, 10)
Test: (5000, 10)


In [7]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [9]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

train_df_clean["label"] = train_df_clean["sentiment"].map(label2id)
val_df_clean["label"] = val_df_clean["sentiment"].map(label2id)
test_df_clean["label"] = test_df_clean["sentiment"].map(label2id)

print(train_df_clean[["text", "sentiment", "label"]].head())

                                                text sentiment  label
0  arrived broken. manufacturer defect. two of th...  negative      0
1  the cabinet dot were all detached from backing...  negative      0
2  i received my first order of this product and ...  negative      0
3  this product is a piece of shit. do not buy. d...  negative      0
4  went through 3 in one day doesn't fit correct ...  negative      0


In [22]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [24]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df_clean[["text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df_clean[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df_clean[["text", "label"]],
    preserve_index=False
)

print(train_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 199499
})


In [25]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_train)

Map: 100%|██████████| 5000/5000 [00:00<00:00, 11191.29 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 199499
})


In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("BERT model loaded successfully!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4056.25it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

BERT model loaded successfully!


In [27]:
import accelerate

print("Accelerate:", accelerate.__version__)

Accelerate: 1.14.0


In [28]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_sentiment_results",

    num_train_epochs=2,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    logging_steps=100,

    fp16=True,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none"
)

print("Training arguments configured!")

Training arguments configured!


In [29]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

print("Trainer ready!")

Trainer ready!


In [32]:
from transformers import EarlyStoppingCallback

In [33]:
num_train_epochs=2

In [34]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_sentiment_results",

    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    logging_steps=100,

    fp16=True,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none"
)

In [35]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    processing_class=tokenizer,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("Trainer ready with Early Stopping!")

Trainer ready with Early Stopping!


In [36]:
load_best_model_at_end=True

In [39]:
train_result = trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1
1000,1.885319,0.675704,0.746549,0.662559
2000,2.327237,0.576091,0.754551,0.701807
3000,2.189919,0.566307,0.762152,0.693039
4000,2.214510,0.580565,0.759352,0.705474
5000,2.255102,0.562303,0.758152,0.693373
6000,2.198577,0.558268,0.769554,0.703110


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.37it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

In [40]:
test_results = trainer.evaluate(tokenized_test)

print(test_results)

Training Loss,Validation Loss,Step,Accuracy,F1
2.198577,0.560950,6000,0.764600,0.710495


{'eval_loss': 0.5609496831893921, 'eval_accuracy': 0.7646, 'eval_f1': 0.7104945232561333}


In [41]:
from sklearn.metrics import classification_report

pred_output = trainer.predict(tokenized_test)

test_preds = np.argmax(
    pred_output.predictions,
    axis=-1
)

print(
    classification_report(
        test_df_clean["label"],
        test_preds,
        target_names=["negative", "neutral", "positive"]
    )
)

              precision    recall  f1-score   support

    negative       0.80      0.84      0.82      2000
     neutral       0.47      0.44      0.45      1000
    positive       0.87      0.85      0.86      2000

    accuracy                           0.76      5000
   macro avg       0.71      0.71      0.71      5000
weighted avg       0.76      0.76      0.76      5000



Experiment 2: Class-Weighted BERT.

In [42]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1, 2])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df_clean["label"]
)

print("Class weights:")
for label, weight in zip(classes, class_weights):
    print(f"{id2label[label]}: {weight:.4f}")

Class weights:
negative: 0.8331
neutral: 1.6646
positive: 0.8341


In [43]:
from transformers import AutoModelForSequenceClassification

weighted_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Weighted BERT model ready!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8229.72it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Weighted BERT model ready!


In [44]:
import torch
from transformers import Trainer

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float
)

if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.to("cuda")

print("Class weights tensor:", class_weights_tensor)

Class weights tensor: tensor([0.8331, 1.6646, 0.8341], device='cuda:0')


In [45]:
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights_tensor
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

print("WeightedTrainer ready!")

WeightedTrainer ready!


In [46]:
from transformers import TrainingArguments

weighted_training_args = TrainingArguments(
    output_dir="./bert_sentiment_weighted_results",

    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    logging_steps=100,

    fp16=True,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none"
)

print("Weighted training arguments configured!")

Weighted training arguments configured!


In [ ]:
from transformers import EarlyStoppingCallback

weighted_trainer = WeightedTrainer(
    model=weighted_model,
    args=weighted_training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    processing_class=tokenizer,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("Weighted Trainer ready!")

Weighted Trainer ready!


In [48]:
train_result_weighted = weighted_trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1
1000,2.679919,0.597378,0.748550,0.696097
2000,2.557097,0.608208,0.724145,0.701827
3000,2.461530,0.592820,0.740148,0.707181
4000,2.454298,0.587772,0.748750,0.714398
5000,2.573028,0.572941,0.745949,0.707280
6000,2.432546,0.567318,0.747349,0.711216


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.03it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

In [49]:
print("Best checkpoint:", weighted_trainer.state.best_model_checkpoint)
print("Best metric:", weighted_trainer.state.best_metric)

Best checkpoint: ./bert_sentiment_weighted_results\checkpoint-4000
Best metric: 0.7143980612130258


In [50]:
weighted_test_results = weighted_trainer.evaluate(tokenized_test)

print(weighted_test_results)

Training Loss,Validation Loss,Step,Accuracy,F1
2.432546,0.571260,6000,0.748800,0.715126


{'eval_loss': 0.5712597370147705, 'eval_accuracy': 0.7488, 'eval_f1': 0.7151264587918623}


In [51]:
from sklearn.metrics import classification_report
import numpy as np

weighted_pred_output = weighted_trainer.predict(tokenized_test)

weighted_test_preds = np.argmax(
    weighted_pred_output.predictions,
    axis=-1
)

print(
    classification_report(
        test_df_clean["label"],
        weighted_test_preds,
        target_names=["negative", "neutral", "positive"],
        digits=4
    )
)

              precision    recall  f1-score   support

    negative     0.8244    0.7630    0.7925      2000
     neutral     0.4333    0.5720    0.4931      1000
    positive     0.8999    0.8230    0.8598      2000

    accuracy                         0.7488      5000
   macro avg     0.7192    0.7193    0.7151      5000
weighted avg     0.7764    0.7488    0.7595      5000



In [53]:
import os


In [54]:
os.makedirs("models/sentiment/bert_weighted", exist_ok=True)

weighted_trainer.save_model(
    "models/sentiment/bert_weighted"
)

tokenizer.save_pretrained(
    "models/sentiment/bert_weighted"
)

print("3-Class Weighted BERT saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.61s/it]

3-Class Weighted BERT saved successfully.
